# Projeto de Machine Learning — Indicador RIPSA (base `ripsa008mb.csv`)

**Objetivo:** construir um pipeline completo de análise preditiva a partir da base de indicadores municipais do RIPSA, cobrindo:

1. Carga e inspeção dos dados
2. Tratamento de dados (nulos, tipos, remoção de vazamento de dado)
3. Engenharia de atributos e definição do alvo
4. Pré-processamento (`ColumnTransformer`)
5. Separação treino/teste
6. Treinamento com `scikit-learn`
7. Avaliação do modelo
8. Exportação dos resultados para o **Power BI**

> Observação sobre a base: cada linha representa o valor de um indicador epidemiológico
> em um **município**, em um **mês/ano** (`co_anomes`), dentro de uma **categoria de
> desagregação** (sexo, faixa etária, situação vacinal, tipo do caso). A grande maioria
> dos registros (~97%) tem valor 0 — ou seja, é um evento raro por município/mês/categoria.


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                              roc_curve, average_precision_score)
import joblib, json

RANDOM_STATE = 42
pd.set_option('display.max_columns', None)


## 1. Carga dos dados

No Colab, faça upload do `ripsa008mb.csv` (ícone de pasta à esquerda > upload) e ajuste o caminho abaixo, ou monte o Google Drive.


In [2]:
# Caminho do arquivo (ajuste conforme o ambiente: Colab, Drive, ou execução local)
CAMINHO_CSV = 'ripsa008mb.csv'

df = pd.read_csv(CAMINHO_CSV)
print('Shape:', df.shape)
df.head()


Shape: (366572, 29)


,co_anomes,co_ibge,vl_indicador_calculado_mun,co_uf,no_municipio,sg_uf,no_uf,co_regiao_brasil,no_regiao_brasil,sg_regiao_brasil,co_regiao_saude,no_regiao_saude,no_macro,co_macro,vl_indicador_calculado_rs,vl_indicador_calculado_ms,vl_indicador_calculado_uf,vl_indicador_calculado_reg,vl_indicador_calculado_br,vl_indicador_calculado_al,dt_competencia,dt_atualizacao,ds_unidade_medida,sg_granularidade,ds_granularidade,sg_categoria,ds_categoria,co_item_categoria,ds_item_categoria
0,202212,350390,0.0,35,Arujá,SP,São Paulo,3,Sudeste,SE,35011,Alto do Tietê,RRAS2,3525,0.0,0.0,0.0,0.0,0.0,0.0,2022-12-01 12:00:00,2026-01-26 00:00:00,Número,MN,Município,FXETC1,Faixa etária,NaN,NaN
1,202212,315570,0.0,31,Rio Piracicaba,MG,Minas Gerais,3,Sudeste,SE,31023,João Monlevade,Centro,3103,0.0,0.0,0.0,0.0,0.0,0.0,2022-12-01 12:00:00,2026-01-26 00:00:00,Número,MN,Município,FXETC1,Faixa etária,NaN,NaN
2,201612,220620,0.0,22,Miguel Alves,PI,Piauí,2,Nordeste,NE,22004,Entre Rios,Meio Norte,2208,0.0,0.0,0.0,0.0,0.0,0.0,2016-12-01 12:00:00,2026-01-26 00:00:00,Número,MN,Município,FXETC1,Faixa etária,NaN,NaN
3,201712,353010,0.0,35,Mirandópolis,SP,São Paulo,3,Sudeste,SE,35022,Lagos do DRS II,RRAS19,3536,0.0,0.0,0.0,0.0,0.0,0.0,2017-12-01 12:00:00,2026-01-26 00:00:00,Número,MN,Município,FXETC1,Faixa etária,NaN,NaN
4,201912,292700,0.0,29,Rio Real,BA,Bahia,2,Nordeste,NE,29001,Alagoinhas,Nordeste (NRS - Alagoinhas),2914,0.0,0.0,0.0,0.0,0.0,0.0,2019-12-01 12:00:00,2026-01-26 00:00:00,Número,MN,Município,FXETC1,Faixa etária,NaN,NaN


## 2. Inspeção inicial

In [3]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 366572 entries, 0 to 366571
Data columns (total 29 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   co_anomes                   366572 non-null  int64  
 1   co_ibge                     366572 non-null  int64  
 2   vl_indicador_calculado_mun  366572 non-null  float64
 3   co_uf                       366572 non-null  int64  
 4   no_municipio                366572 non-null  str    
 5   sg_uf                       366572 non-null  str    
 6   no_uf                       366572 non-null  str    
 7   co_regiao_brasil            366572 non-null  int64  
 8   no_regiao_brasil            366572 non-null  str    
 9   sg_regiao_brasil            366572 non-null  str    
 10  co_regiao_saude             366572 non-null  int64  
 11  no_regiao_saude             366572 non-null  str    
 12  no_macro                    366572 non-null  str    
 13  co_macro                 

In [4]:
df.isnull().sum().sort_values(ascending=False)


ds_item_categoria             284700
co_item_categoria              71175
vl_indicador_calculado_mun         0
co_ibge                            0
co_anomes                          0
sg_uf                              0
no_uf                              0
co_regiao_brasil                   0
no_regiao_brasil                   0
sg_regiao_brasil                   0
co_regiao_saude                    0
co_uf                              0
no_municipio                       0
no_macro                           0
no_regiao_saude                    0
co_macro                           0
vl_indicador_calculado_rs          0
vl_indicador_calculado_reg         0
vl_indicador_calculado_br          0
vl_indicador_calculado_ms          0
vl_indicador_calculado_uf          0
dt_competencia                     0
vl_indicador_calculado_al          0
dt_atualizacao                     0
ds_unidade_medida                  0
ds_granularidade                   0
sg_granularidade                   0
d

## 3. Tratamento de dados

### 3.1 Datas
`co_anomes` vem no formato `AAAAMM` (ex.: 202212). Extraímos `ano` e `mes` como atributos numéricos.

### 3.2 Vazamento de dado (data leakage)
Duas famílias de colunas foram identificadas como **vazamento** e precisam ser removidas antes de treinar o modelo:

- **`vl_indicador_calculado_rs/ms/uf/reg/br/al`**: são valores agregados (região de
  saúde, macrorregião, UF, região, Brasil, "al") que **somam o próprio valor do
  município** que queremos prever — usá-los como *feature* seria vazar a resposta.
- **`co_item_categoria` / `ds_item_categoria`**: ao cruzar com o alvo, verificamos que
  esses campos **só são preenchidos quando o indicador do município é maior que zero**
  (quando é nulo, o alvo é sempre 0). Isso é vazamento perfeito e também precisa ser
  descartado das features.

Essas colunas foram mantidas apenas no dataset de identificação (para consulta no
Power BI), nunca como entrada do modelo.


In [5]:
df['co_anomes'] = df['co_anomes'].astype(str)
df['ano'] = df['co_anomes'].str[:4].astype(int)
df['mes'] = df['co_anomes'].str[4:6].astype(int)

cols_vazamento = ['vl_indicador_calculado_rs', 'vl_indicador_calculado_ms',
                   'vl_indicador_calculado_uf', 'vl_indicador_calculado_reg',
                   'vl_indicador_calculado_br', 'vl_indicador_calculado_al',
                   'co_item_categoria', 'ds_item_categoria']

cols_identificacao = ['co_ibge', 'no_municipio', 'sg_uf', 'no_uf', 'co_uf',
                       'co_regiao_brasil', 'no_regiao_brasil', 'sg_regiao_brasil',
                       'co_regiao_saude', 'no_regiao_saude', 'no_macro', 'co_macro',
                       'dt_competencia', 'dt_atualizacao', 'ds_unidade_medida',
                       'sg_granularidade', 'ds_granularidade']


## 4. Definição da variável-alvo

Como o indicador é um evento raro (~97% dos registros são 0), transformamos o problema
em **classificação binária**: o município registrou ao menos 1 caso naquele mês/categoria?

`teve_caso = 1` se `vl_indicador_calculado_mun > 0`, senão `0`.


In [6]:
df['teve_caso'] = (df['vl_indicador_calculado_mun'] > 0).astype(int)
print('Taxa da classe positiva:', round(df['teve_caso'].mean(), 4))
df = df.drop(columns=cols_vazamento)
df['teve_caso'].value_counts(normalize=True)


Taxa da classe positiva: 0.0292


teve_caso
0    0.970819
1    0.029181
Name: proportion, dtype: float64

## 5. Seleção de atributos (features)

Mantemos apenas atributos **conhecidos antes do evento** e sem vazamento:

- **Numéricos:** `ano`, `mes`
- **Categóricos:** `sg_uf` (UF), `no_regiao_brasil` (região), `ds_categoria`
  (dimensão de desagregação do indicador: sexo, faixa etária, situação vacinal, tipo do caso)

Códigos mais granulares (`co_regiao_saude`, `co_macro`, `no_municipio`) foram
propositalmente deixados de fora das features para evitar que o modelo memorize o
município (o que geraria overfitting e não generalizaria para dados novos), mas
seguem disponíveis no CSV de saída para segmentação no Power BI.


In [7]:
features_numericas = ['ano', 'mes']
features_categoricas = ['sg_uf', 'no_regiao_brasil', 'ds_categoria']

X = df[features_numericas + features_categoricas].copy()
y = df['teve_caso'].copy()


## 6. Separação treino/teste

Usamos `stratify=y` para manter a mesma proporção da classe rara em treino e teste.


In [8]:
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, df.index, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print('Treino:', X_train.shape, ' Teste:', X_test.shape)


Treino: (293257, 5)  Teste: (73315, 5)


## 7. Pré-processamento + Modelo (Pipeline scikit-learn)

- `StandardScaler` para os atributos numéricos (`ano`, `mes`)
- `OneHotEncoder` para os atributos categóricos
- `RandomForestClassifier` com `class_weight='balanced'` para compensar o forte
  desbalanceamento entre as classes

Tudo encapsulado em um único `Pipeline`, o que evita vazamento entre treino/teste e
facilita salvar/reaproveitar o modelo depois.


In [9]:
preprocessador = ColumnTransformer(transformers=[
    ('num', StandardScaler(), features_numericas),
    ('cat', OneHotEncoder(handle_unknown='ignore'), features_categoricas)
])

modelo = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)

pipeline = Pipeline(steps=[
    ('preprocessador', preprocessador),
    ('modelo', modelo)
])

pipeline.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessador', ...), ('modelo', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers

## 8. Avaliação do modelo

In [10]:
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
print('Matriz de confusão:\n', cm)

auc = roc_auc_score(y_test, y_proba)
ap = average_precision_score(y_test, y_proba)
print('ROC-AUC:', round(auc, 4))
print('Average Precision (PR-AUC):', round(ap, 4))


              precision    recall  f1-score   support

           0       1.00      0.90      0.95     71176
           1       0.21      0.91      0.35      2139

    accuracy                           0.90     73315
   macro avg       0.61      0.91      0.65     73315
weighted avg       0.97      0.90      0.93     73315

Matriz de confusão:
 [[64009  7167]
 [  185  1954]]
ROC-AUC: 0.9608
Average Precision (PR-AUC): 0.573


**Leitura dos resultados:** dado o forte desbalanceamento (~2,9% de casos positivos),
priorizamos **recall** da classe rara via `class_weight='balanced'` — o modelo captura a
maior parte dos casos reais (recall alto), ao custo de mais falsos positivos (precisão
menor). Esse é um trade-off típico em vigilância epidemiológica, onde deixar de detectar
um caso real (falso negativo) costuma ser mais custoso do que investigar um falso alarme.
Se a prioridade do negócio for o oposto, ajuste `class_weight`, o limiar de decisão
(`threshold`) sobre `y_proba`, ou compare com outros algoritmos (ex.: Gradient Boosting).


## 9. Importância dos atributos

In [11]:
ohe = pipeline.named_steps['preprocessador'].named_transformers_['cat']
nomes_cat = ohe.get_feature_names_out(features_categoricas)
nomes_features = features_numericas + list(nomes_cat)

importancias = pipeline.named_steps['modelo'].feature_importances_
df_import = pd.DataFrame({'feature': nomes_features, 'importancia': importancias})
df_import = df_import.sort_values('importancia', ascending=False)
df_import.head(15)


,feature,importancia
0,ano,0.605958
27,sg_uf_SP,0.057532
31,no_regiao_brasil_Norte,0.034024
12,sg_uf_MG,0.029684
7,sg_uf_CE,0.027705
15,sg_uf_PA,0.026364
17,sg_uf_PE,0.024974
32,no_regiao_brasil_Sudeste,0.019985
30,no_regiao_brasil_Nordeste,0.019000
28,sg_uf_TO,0.015005


## 10. Exportação para Power BI

Geramos três arquivos prontos para importar no Power BI:

- **`predicoes_powerbi.csv`**: uma linha por registro do conjunto de teste, com
  identificação geográfica/temporal, valor real (`teve_caso`), predição do modelo
  (`predito`), probabilidade (`probabilidade_caso`) e se o modelo acertou (`acerto`).
  Essa é a tabela principal do relatório (mapas por UF/região, taxa de acerto por
  categoria, etc.).
- **`importancia_atributos.csv`**: importância de cada atributo no modelo, para um
  gráfico de barras no Power BI.
- **`curva_roc.csv`**: pontos da curva ROC (FPR x TPR), para visualizar a performance
  do classificador em diferentes limiares.
- **`metricas.csv`**: resumo das métricas do modelo em uma única linha (para um cartão
  de indicador no Power BI).


In [12]:
saida = df.loc[idx_test, cols_identificacao + ['ano', 'mes', 'ds_categoria', 'teve_caso']].copy()
saida['predito'] = y_pred
saida['probabilidade_caso'] = y_proba.round(4)
saida['acerto'] = (saida['teve_caso'] == saida['predito'])
saida.to_csv('predicoes_powerbi.csv', index=False, encoding='utf-8-sig')

df_import.to_csv('importancia_atributos.csv', index=False, encoding='utf-8-sig')

fpr, tpr, _ = roc_curve(y_test, y_proba)
pd.DataFrame({'fpr': fpr, 'tpr': tpr}).to_csv('curva_roc.csv', index=False, encoding='utf-8-sig')

report = classification_report(y_test, y_pred, output_dict=True)
metricas = {
    'acuracia': report['accuracy'],
    'precisao_classe_1': report['1']['precision'],
    'recall_classe_1': report['1']['recall'],
    'f1_classe_1': report['1']['f1-score'],
    'roc_auc': auc,
    'average_precision': ap,
    'n_treino': len(X_train),
    'n_teste': len(X_test),
    'taxa_classe_positiva': float(y.mean())
}
pd.DataFrame([metricas]).to_csv('metricas.csv', index=False, encoding='utf-8-sig')

# Salva o pipeline treinado (pré-processamento + modelo) para reuso futuro
joblib.dump(pipeline, 'pipeline_modelo.joblib')

print('Arquivos gerados: predicoes_powerbi.csv, importancia_atributos.csv, curva_roc.csv, metricas.csv, pipeline_modelo.joblib')


Arquivos gerados: predicoes_powerbi.csv, importancia_atributos.csv, curva_roc.csv, metricas.csv, pipeline_modelo.joblib


## 11. Como usar no Power BI

1. Abra o Power BI Desktop → **Obter Dados** → **Texto/CSV**.
2. Importe os quatro arquivos `.csv` gerados acima como tabelas separadas.
3. Sugestões de visuais:
   - **Mapa ou gráfico de barras por UF** usando `predicoes_powerbi.csv` (`sg_uf`,
     `probabilidade_caso`, `teve_caso`).
   - **Cartões de indicador** com `metricas.csv` (ROC-AUC, recall, precisão).
   - **Gráfico de linha** com `curva_roc.csv` (`fpr` no eixo X, `tpr` no eixo Y) para a
     curva ROC.
   - **Gráfico de barras horizontal** com `importancia_atributos.csv` ordenado por
     `importancia`.
   - Uma medida DAX simples de acurácia por categoria:
     `Taxa de Acerto = DIVIDE(COUNTROWS(FILTER(predicoes_powerbi, predicoes_powerbi[acerto] = TRUE())), COUNTROWS(predicoes_powerbi))`
